# Pandas API on Spark

## Przygotowania

Na początku pobierz dane, które będziemy przetwarzali 

In [1]:
%%sh
wget -q https://jankiewicz.pl/bigdata/bigdata-sp/netflix.zip

Następnie je rozpakuj 

In [2]:
%%sh
unzip netflix.zip

Archive:  netflix.zip
  inflating: movie_titles.csv        
  inflating: rates/part-00.csv       
  inflating: rates/part-01.csv       
  inflating: rates/part-02.csv       
  inflating: rates/part-03.csv       
  inflating: rates/part-04.csv       
  inflating: rates/part-05.csv       
  inflating: rates/part-06.csv       
  inflating: rates/part-07.csv       
  inflating: rates/part-08.csv       
  inflating: rates/part-09.csv       
  inflating: rates/part-10.csv       
  inflating: rates/part-11.csv       


Załaduj je do systemu plików HDFS

In [3]:
%%sh
hadoop fs -put movie_titles.csv /tmp/
hadoop fs -put rates /tmp/

Sprawdź czy są dostępne w ich źródłowych (dla naszego przetwarzania) miejscach 

In [3]:
%%sh
hadoop fs -ls /tmp

Found 3 items
drwx-wx-wx   - hadoop supergroup          0 2025-12-02 20:00 /tmp/hive
-rw-r--r--   2 hadoop supergroup     577561 2025-12-02 20:49 /tmp/movie_titles.csv
drwxr-xr-x   - hadoop supergroup          0 2025-12-02 20:49 /tmp/rates


Podglądnij jak wygląda ich budowa.

Na początek plik z tytułami filmów. Plik ma format CSV. Pola, które są w nim dostępne to:
- `ID` - identyfikator filmu, 
- `Year` - rok premiery
- `Title` - tytuł filmu

In [4]:
%%sh
hadoop fs -head /tmp/movie_titles.csv

ID,Year,Title
1,2003,Dinosaur Planet
2,2004,Isle of Man TT 2004 Review
3,1997,Character
4,1994,Paula Abdul's Get Up & Dance
5,2004,The Rise and Fall of ECW
6,1997,Sick
7,1992,8 Man
8,2004,What the #$*! Do We Know!?
9,1991,Class of Nuke 'Em High 2
10,2001,Fighter
11,1999,Full Frame: Documentary Shorts
12,1947,My Favorite Brunette
13,2003,Lord of the Rings: The Return of the King: Extended Edition: Bonus Material
14,1982,Nature: Antarctica
15,1988,Neil Diamond: Greatest Hits Live
16,1996,Screamers
17,2005,7 Seconds
18,1994,Immortal Beloved
19,2000,By Dawn's Early Light
20,1972,Seeta Aur Geeta
21,2002,Strange Relations
22,2000,Chump Change
23,2001,Clifford: Clifford Saves the Day! / Clifford's Fluffiest Friend Cleo
24,1981,My Bloody Valentine
25,1997,Inspector Morse 31: Death Is Now My Neighbour
26,2004,Never Die Alone
27,1962,Sesame Street: Elmo's World: The Street We Live On
28,2002,Lilo and Stitch
29,2001,Boycott
30,2003,Something's Gotta Give
31,1999,Classic Albums: Meat Loaf: Bat Out

Drugi ze zbiorów danych zawiera oceny filmów dokonane przez użytkowników portali Netflix. Plik także mają format CSV, a dostępne pola są następujące:
- `date` - data wystawienia oceny
- `film_id` - identyfikator filmu
- `user_id` - identyfikator użytkownika
- `rate` - ocena 

In [5]:
%%sh
hadoop fs -head /tmp/rates/part-00.csv

date,film_id,user_id,rate
1999-11-11,1367,510180,5
1999-11-11,1798,510180,5
1999-11-11,2866,510180,3
1999-11-11,3730,510180,4
1999-11-11,3870,510180,2
1999-11-11,5571,510180,4
1999-11-11,5625,510180,4
1999-11-11,6615,510180,5
1999-11-11,8079,510180,2
1999-11-11,8357,510180,4
1999-11-11,8651,510180,2
1999-11-11,9003,510180,3
1999-11-11,9392,510180,3
1999-11-11,9798,510180,3
1999-11-11,10341,510180,4
1999-11-11,10774,510180,3
1999-11-11,11234,510180,5
1999-11-11,11313,510180,2
1999-11-11,12470,510180,2
1999-11-11,12473,510180,5
1999-11-11,12818,510180,2
1999-11-11,14660,510180,2
1999-11-11,15057,510180,5
1999-11-11,15105,510180,4
1999-11-11,15894,510180,3
1999-11-11,16465,510180,3
1999-11-11,17064,510180,2
1999-11-11,17764,510180,5
1999-12-06,2478,510180,3
1999-12-06,2948,510180,3
1999-12-06,3421,510180,3
1999-12-06,5474,510180,2
1999-12-06,6902,510180,4
1999-12-06,9536,510180,5
1999-12-06,11080,510180,3
1999-12-06,11259,510180,3
1999-12-06,11612,510180,3
1999-12-06,14455,510180,3
1999-1

i jeszcze mała korekta biblioteki `NumPy`, gdyż wersje 2.0 i wyższe nie są w pełni kompatybilne z *Pandas API on Spark* dla wersji Sparka 3.5.5 i niższych

In [6]:
pip install "numpy<2.0"

Note: you may need to restart the kernel to use updated packages.


## Odczyt danych

Na początek kontekst. Zanim jednak to zrobimy wywołamy pomocniczy moduł `findspark`. 
Jego zadania są następujące:
- Dodaje *Sparka* do ścieżki Pythona (`sys.path`) — dzięki czemu możesz zaimportować `pyspark` w środowiskach, które normalnie nie mają ustawionych zmiennych środowiskowych Sparka.
- Automatycznie wykrywa lokalizację instalacji *Sparka* (jeśli jest zainstalowany standardowo) lub możesz mu wskazać ścieżkę ręcznie `findspark.init("/path/to/spark")`
- Ustawia wymagane zmienne środowiskowe, np. `SPARK_HOME`, aby `pyspark` działał poprawnie.

In [7]:
import findspark
findspark.init()

a teraz kontekst

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

25/12/02 20:57:05 INFO SparkEnv: Registering MapOutputTracker
25/12/02 20:57:05 INFO SparkEnv: Registering BlockManagerMaster
25/12/02 20:57:05 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/12/02 20:57:05 INFO SparkEnv: Registering OutputCommitCoordinator
25/12/02 20:57:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Ustawmy stosowne zmienne środowiskowe, a następnie zaimportujmy potrzebne biblioteki

In [9]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

In [10]:
import pyspark.pandas as ps

Zaczytajmy dane do *Pandas Dataframe on Spark*

Na początku informacje na temat filmów

In [22]:
spark.conf.set("spark.sql.ansi.enabled", "false")

In [23]:
movies_ps = ps.read_csv('/tmp/movie_titles.csv')

/opt/spark/python/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


Dowiedzmy się jak wygląda "budowa" naszych danych

In [24]:
movies_ps.shape

(17770, 3)

In [25]:
movies_ps.columns

Index(['ID', 'Year', 'Title'], dtype='object')

In [26]:
movies_ps.head()

,ID,Year,Title
0,1,2003,Dinosaur Planet
1,2,2004,Isle of Man TT 2004 Review
2,3,1997,Character
3,4,1994,Paula Abdul's Get Up & Dance
4,5,2004,The Rise and Fall of ECW


Czas zatem na oceny

In [27]:
rates_ps = ps.read_csv('/tmp/rates')

/opt/spark/python/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


Dowiedzmy się jak wygląda "budowa" naszych danych

In [28]:
rates_ps.shape

(14780972, 4)

In [29]:
rates_ps.columns

Index(['date', 'film_id', 'user_id', 'rate'], dtype='object')

In [30]:
rates_ps.head()

,date,film_id,user_id,rate
0,2003-04-07,1256,406841,4
1,2003-04-07,1256,79527,4
2,2003-04-07,1256,108859,4
3,2003-04-07,1260,1816643,3
4,2003-04-07,1260,1827071,3


To co, jesteśmy w domu? Zatem do dzieła. Trzy proste zadania

In [33]:
movies_ps.head()

,ID,Year,Title
0,1,2003,Dinosaur Planet
1,2,2004,Isle of Man TT 2004 Review
2,3,1997,Character
3,4,1994,Paula Abdul's Get Up & Dance
4,5,2004,The Rise and Fall of ECW


# Zadania

## Zadanie 1

Określ ile filmów nie uzyskało żadnej oceny. Utwórz *Pandas Dataframe on Spark*, który będzie zawierał komplet informacji filmów ograniczonych tylko do tych, które mają chociaż jedną ocenę.  

In [55]:
joined_dataframe = ps.merge(rates_ps, movies_ps, left_on=["film_id"], right_on=["ID"], how="right")[["ID","rate"]]
rate_count = joined_dataframe.groupby("ID").agg("count")
movies_without_rate = rate_count[rate_count["rate"] == 0]
print(f"Liczba filmów bez oceny: {str(len(movies_without_rate))}")

[Stage 58:==============================================>           (4 + 1) / 5]

Liczba filmów bez oceny: 6056


## Zadanie 2

Dla każdej dekady premiery filmów utwórz dwa rankingi

- trzech filmów o największej liczbie ocen
- trzech filmów o największej średniej ocen

In [61]:
# Merge dataframes
merged_df = rates_ps.merge(movies_ps, left_on='film_id', right_on='ID', how='inner')

# Konwersja daty
merged_df['date'] = ps.to_datetime(merged_df['date'])

# Konwersja roku na numeric
merged_df['Year'] = ps.to_numeric(merged_df['Year'], errors='coerce')

# Obliczenie dekady
merged_df['decade'] = (merged_df['Year'] // 10) * 10

# Top 3 filmy po liczbie ocen
top_movies_by_count = (
    merged_df.groupby(['decade', 'Title'])
    .agg({'rate': 'count'})
    .reset_index()
    .sort_values(['decade', 'rate'], ascending=[True, False])
    .groupby('decade')
    .head(3)
)

# Top 3 filmy po średniej ocen
top_movies_by_avg = (
    merged_df.groupby(['decade', 'Title'])
    .agg({'rate': 'mean'})
    .reset_index()
    .sort_values(['decade', 'rate'], ascending=[True, False])
    .groupby('decade')
    .head(3)
)

print(f"Ranking oparty na największej liczbie ocen:\n {top_movies_by_count}")
print(f"Ranking oparty na największej średniej ocen:\n {top_movies_by_avg}")

Ranking oparty na największej liczbie ocen:
        decade                                        Title   rate
3072   1890.0                Lumiere Brothers' First Films     30
11452  1900.0  D.W. Griffith: Years of Discovery 1909-1913      8
3645   1910.0                        The Birth of a Nation    611
9916   1910.0                                           20    489
163    1910.0                  The Cabinet of Dr. Caligari    457
1627   1920.0                            The Mark of Zorro   1951
10380  1920.0                     The Last of the Mohicans   1347
3550   1920.0                                   Metropolis    756
60     1930.0        The Wizard of Oz: Collector's Edition  18318
2310   1930.0      Gone with the Wind: Collector's Edition  12098
3505   1930.0                 Mr. Smith Goes to Washington   5682
2431   1940.0                        It's a Wonderful Life  12897
2340   1940.0                                 Citizen Kane  11105
5390   1940.0                  

Ranking oparty na największej średniej ocen:
        decade                                            Title      rate
3121   1890.0                    Lumiere Brothers' First Films  3.700000
9829   1900.0      D.W. Griffith: Years of Discovery 1909-1913  2.875000
204    1910.0                      The Cabinet of Dr. Caligari  3.575492
4520   1910.0                                      Intolerance  3.406250
10483  1910.0                                               20  3.312883
8143   1920.0                       The Passion of Joan of Arc  4.011450
10055  1920.0                             The General (Silent)  3.878924
86     1920.0                              Battleship Potemkin  3.863636
11096  1930.0                                     Libeled Lady  5.000000
11171  1930.0                             Mutiny on the Bounty  5.000000
11178  1930.0          Imitation of Life: Double Feature (1934  4.500000
11184  1940.0                                     Battleground  5.000000
11256

## Zadanie 3

Dla trzech wybranych użytkowników, wśród tych, którzy ocenili co najemniej 10 filmów, wyznacz trzy najbardziej rekomendowane filmy do obejrzenia. 

Wskaż źródła opisujące zastosowane przez Ciebie podejście. 

Koniecznie uzupełnij kod szczegółowymi komentarzami. 